# Group 4: feature processing notebook

Этот ноутбук предназначен для исследовательской обработки признаков группы `4`.

Логика ячеек специально повторяет структуру `src/features/group_4/feature_processor.py`,
чтобы позже перенос был почти механическим.

In [1]:
from pathlib import Path
import pandas as pd
import sys

# путь до корня проекта
PROJECT_ROOT = Path("../../").resolve()

# добавляем в PYTHONPATH
sys.path.append(str(PROJECT_ROOT))

# теперь обычный импорт
from src.utils.spec_converter import create_feature_spec_template
from src.utils.io import load_feature_names_from_txt

In [2]:
# Пути относительно папки notebooks/group_4/
DATA_PATH = Path("../../data/raw/MIPT_hackathon_dataset.csv")
FEATURES_PATH = Path("../../data/feature_groups/features_group_4.txt")
OUTPUT_DIR = Path("../../notebook_outputs/group_4")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
df = pd.read_csv(DATA_PATH)
feature_names = load_feature_names_from_txt(FEATURES_PATH)
block_df = df[feature_names].copy()

print("Block shape:", block_df.shape)
display(block_df.head())

Block shape: (5383, 19)


,contact_Число сделок,lead_Состав заказа,lead_source,lead_Нумерация сделки,contact_Адрес ПВЗ,days_to_outcome,lead_Линейная ширина (см),lead_Сумма заказа,lead_Скидка,lead_status_id,closed_ts,contact_updated_at,contact_id,lead_Список товаров GoSklad,lead_Тип отправления,days_sale_to_handed,lead_updated_at,sale_ts,lead_Дата приобретения изделия
0,2.0,1) Тапки р-р 35-42\nАртикул: 24\nКол-во: 1 шт\...,NaN,NaN,"Ул.Планерная, 63/1",7.26,NaN,NaN,15.0,142,1.762602e+09,1771091954,CTR_0964,NaN,NaN,3.10,1771091802,1761974724,NaN
1,1.0,1) Подушка для реабилитации и лечения восьмимо...,NaN,NaN,"ул. Крупской, 60а",9.42,NaN,NaN,NaN,142,1.762789e+09,1771089448,CTR_0957,NaN,NaN,3.10,1771089174,1761975068,NaN
2,6.0,1) Подушка для реабилитации и лечения восьмимо...,NaN,NaN,NaN,8.94,NaN,NaN,NaN,142,1.762749e+09,1771089347,CTR_0229,NaN,NaN,3.18,1771088926,1761976628,NaN
3,NaN,1) Подушка для реабилитации и лечения двенадца...,NaN,NaN,"пр-т Ленина, 6",101.30,NaN,NaN,NaN,143,1.770729e+09,1771089448,CTR_0967,NaN,NaN,3.08,1771089174,1761976896,NaN
4,1.0,1) Подушка для реабилитации и лечения с против...,NaN,NaN,NaN,7.01,NaN,NaN,NaN,142,1.762583e+09,1771090234,CTR_0882,NaN,NaN,3.10,1771314071,1761977233,NaN


## Функции обработки признаков

Названия функций совпадают с private-методами из `feature_processor.py`.

In [ ]:
def _add_width_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "lead_Ширина" not in block_df.columns:
        return
    series = pd.to_numeric(block_df["lead_Ширина"], errors="coerce")
    result["lead_Ширина"] = series.fillna(series.median())


def _add_linear_height_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "lead_Линейная высота (см)" not in block_df.columns:
        return
    series = pd.to_numeric(block_df["lead_Линейная высота (см)"], errors="coerce")
    result["lead_Линейная высота (см)"] = series.fillna(series.median())
    result["lead_Линейная высота (см)__was_missing"] = series.isna().astype(int)


def _add_payment_type_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "lead_Вид оплаты" not in block_df.columns:
        return
    series = (
        block_df["lead_Вид оплаты"]
        .astype("string")
        .fillna("UNKNOWN")
        .str.strip()
        .str.lower()
    )
    result["lead_Вид оплаты"] = series.replace({"": "unknown"}).astype(str)


def _add_returned_ts_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "returned_ts" not in block_df.columns:
        return
    ts = pd.to_datetime(block_df["returned_ts"], errors="coerce")
    result["returned_ts"] = ts.astype("string")
    result["returned_ts__is_present"] = ts.notna().astype(int)


def _add_delivery_service_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "lead_Служба доставки" not in block_df.columns:
        return
    series = (
        block_df["lead_Служба доставки"]
        .astype("string")
        .fillna("UNKNOWN")
        .str.strip()
    )
    value_counts = series.value_counts(dropna=False)
    rare_values = value_counts[value_counts < 10].index
    result["lead_Служба доставки"] = series.replace(list(rare_values), "OTHER").astype(str)


def _add_default_feature(block_df: pd.DataFrame, result: pd.DataFrame, column: str) -> None:
    series = block_df[column]
    if pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series):
        result[column] = series.fillna("").astype(str)
    else:
        result[column] = series

In [ ]:
X_block = pd.DataFrame(index=block_df.index)

for column in block_df.columns:
    if column == "lead_Ширина":
        _add_width_feature(block_df, X_block)
    elif column == "lead_Линейная высота (см)":
        _add_linear_height_feature(block_df, X_block)
    elif column == "lead_Вид оплаты":
        _add_payment_type_feature(block_df, X_block)
    elif column == "returned_ts":
        _add_returned_ts_feature(block_df, X_block)
    elif column == "lead_Служба доставки":
        _add_delivery_service_feature(block_df, X_block)
    else:
        _add_default_feature(block_df, X_block, column)

print("Processed shape:", X_block.shape)
display(X_block.head())

In [ ]:
X_block.to_csv(OUTPUT_DIR / "X_block.csv", index=False)
feature_spec = create_feature_spec_template(X_block)
feature_spec.to_csv(OUTPUT_DIR / "feature_spec.csv", index=False)

print("Saved:")
print(OUTPUT_DIR / "X_block.csv")
print(OUTPUT_DIR / "feature_spec.csv")